# Análise de dados TCP-CII

### Importação dos parâmetros universais

In [ ]:
from pathlib import Path
import importlib.util

path = Path("../../../parametros/config.py").resolve()

spec = importlib.util.spec_from_file_location("parametros", path)
parametros = importlib.util.module_from_spec(spec)
spec.loader.exec_module(parametros)

In [ ]:
# Parâmetros importados do arquivo config.py
print("Filtrar por quantidade de alelos TCC2:.........................", parametros.filtarar_por_qte_de_alelos_tcc2)
print("Parâmetro de filtragem median binding percentile TCC2:.........", parametros.parametro_de_filtragem_mbp_tcc2)
print("Percentual de match mínimo TCC2:...............................", parametros.percent_match_minimo_tcc2)

Filtrar por quantidade de alelos TCC1:......................... 10
Parâmetro de filtragem median binding percentile TCC1:......... 5
Percentual de match mínimo TCC1:............................... 95.0


In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('./T CELL/DENV 4 - T Cell Prediction - Class II.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhciipan_el core,netmhciipan_el score,netmhciipan_el percentile
0,1,KNQTWQIEKASLIEVK,206,221,16,HLA-DRB1*01:01,246,0.01,WQIEKASLI,0.986066,0.01
1,1,KNQTWQIEKASLIEVKT,206,222,17,HLA-DRB1*01:01,314,0.04,WQIEKASLI,0.978820,0.04
2,1,KNQTWQIEKASLIEV,206,220,15,HLA-DRB1*01:01,178,0.06,WQIEKASLI,0.974156,0.06
3,1,KNQTWQIEKASLIEVKTC,206,223,18,HLA-DRB1*01:01,382,0.06,WQIEKASLI,0.961404,0.06
4,1,WIESSKNQTWQIEKASLIEVK,201,221,21,HLA-DRB1*01:01,582,0.07,WQIEKASLI,0.829909,0.07
...,...,...,...,...,...,...,...,...,...,...,...
16411,1,TTASGKLVTQWCCR,301,314,14,HLA-DRB3*02:02,129,100.00,SGKLVTQWC,0.000011,100.00
16412,1,TTASGKLVTQWCCRSCTMPP,301,320,20,HLA-DRB3*02:02,535,100.00,LVTQWCCRS,0.000011,100.00
16413,1,KLVTQWCCRSCTMPPLRFLGE,306,326,21,HLA-DRB1*04:01,603,100.00,CRSCTMPPL,0.000010,100.00
16414,1,TTASGKLVTQWCC,301,313,13,HLA-DRB3*02:02,61,100.00,TASGKLVTQ,0.000006,100.00


## Selecionando Epítopos por median binding percentile.

In [ ]:
mbp_minimo = parametros.parametro_de_filtragem_mbp_tcc2

In [ ]:
df_mbp_m5 = df[df['median binding percentile'] < mbp_minimo].copy()
print("Filtrando por median binding percentile < ", mbp_minimo)
df_mbp_m5

Filtrando por median binding percentile <  5


,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,ASGKLVTQW,303,311,9,HLA-B*57:01,303,0.01,ASGKLVTQW,ASGKLVTQW,0.993994,0.01
1,1,ASGKLVTQW,303,311,9,HLA-B*58:01,303,0.01,ASGKLVTQW,ASGKLVTQW,0.993016,0.01
2,1,ITNELNYVLW,71,80,10,HLA-B*57:01,415,0.01,ITNELNYLW,ITNELNYVLW,0.989226,0.01
3,1,SQMLIPKSY,239,247,9,HLA-B*15:01,239,0.01,SQMLIPKSY,SQMLIPKSY,0.980178,0.01
4,1,IESSKNQTW,202,210,9,HLA-B*44:02,202,0.01,IESSKNQTW,IESSKNQTW,0.978499,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
2606,1,CLWPKTHTLW,223,232,10,HLA-B*44:03,567,4.90,CLWPKTHLW,CLWPKTHTLW,0.003115,4.90
2607,1,NELNYVLWE,73,81,9,HLA-B*44:03,73,4.90,NELNYVLWE,NELNYVLWE,0.003085,4.90
2608,1,DQKAVHADMGY,190,200,11,HLA-B*44:02,877,4.90,DQKAVHMGY,DQKAVHADMGY,0.002847,4.90
2609,1,TPPVSDLKY,105,113,9,HLA-B*44:02,105,4.90,TPPVSDLKY,TPPVSDLKY,0.002830,4.90


## Agrupando por pepitideos e agregando colunas pertinentes.

In [4]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    ))

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIKDQKAVHADM,186,198,2,0.745,"HLA-DRB1*01:01, HLA-DRB4*01:01"
1,AAIKDQKAVHADMG,186,199,1,0.720,HLA-DRB4*01:01
2,AKIFTPEARNSTF,121,133,1,0.800,HLA-DRB1*11:01
3,ERRAWNSLEVEDY,146,158,1,0.490,HLA-DQA1*05:01/DQB1*02:01
4,ERRAWNSLEVEDYG,146,159,1,0.260,HLA-DQA1*05:01/DQB1*02:01
...,...,...,...,...,...,...
75,YRQGYATQTVGPWHLGK,256,272,1,0.520,HLA-DQA1*05:01/DQB1*02:01
76,YRQGYATQTVGPWHLGKL,256,273,1,0.510,HLA-DQA1*05:01/DQB1*02:01
77,YRQGYATQTVGPWHLGKLE,256,274,1,0.660,HLA-DQA1*05:01/DQB1*02:01
78,YRQGYATQTVGPWHLGKLEI,256,275,1,0.820,HLA-DQA1*05:01/DQB1*02:01


## Filtragem por qte_de_alelos

In [ ]:
# qte_de_alelos_minima = parametros.filtarar_por_qte_de_alelos_tcc2
qte_de_alelos_minima = 2

In [ ]:
# Filtro do número de alelos
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= qte_de_alelos_minima
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,ATRLENIMW,60,68,13,3.300,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
1,AVHADMGYW,193,201,10,1.645,"HLA-A*23:01, HLA-A*26:01, HLA-A*30:02, HLA-A*3..."
2,CIWPKSHTL,223,231,20,1.600,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2..."
3,CTLPPLRFK,316,324,12,1.850,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
4,ECPDNQRAW,142,150,11,1.900,"HLA-A*23:01, HLA-A*24:02, HLA-A*26:01, HLA-A*3..."
5,ESEMIIPKIY,238,247,10,2.550,"HLA-A*01:01, HLA-A*26:01, HLA-A*30:02, HLA-B*3..."
6,ETWKLARASF,208,217,13,2.800,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
7,EVHTWTEQY,24,32,16,1.500,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
8,EVHTWTEQYKF,24,34,10,3.250,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
9,FQADSPKRL,34,42,18,2.450,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2..."


In [5]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,KNQTWQIEKASLIEV,206,220,1,0.06,HLA-DRB1*01:01
1,KNQTWQIEKASLIEVKTC,206,223,1,0.06,HLA-DRB1*01:01
2,WIESSKNQTWQIEKASLIEVK,201,221,1,0.07,HLA-DRB1*01:01
3,KNQTWQIEKASLIEVKTCL,206,224,1,0.08,HLA-DRB1*01:01
4,KNQTWQIEKASLIEVKTCLW,206,225,1,0.11,HLA-DRB1*01:01
...,...,...,...,...,...,...
75,YRQGYATQTVGPWHLGKLEID,256,276,1,0.86,HLA-DQA1*05:01/DQB1*02:01
76,HRLMSAAIKDQKAVHADMG,181,199,1,0.87,HLA-DRB4*01:01
77,GSGIFVVDNVHTWTEQYKF,16,34,1,0.90,HLA-DRB3*01:01
78,QKAVHADMGYWIES,191,204,1,0.95,HLA-DRB3*01:01


In [6]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,KNQTWQIEKASLIEV,206,220,1,0.06,HLA-DRB1*01:01
1,KNQTWQIEKASLIEVKTC,206,223,1,0.06,HLA-DRB1*01:01
2,WIESSKNQTWQIEKASLIEVK,201,221,1,0.07,HLA-DRB1*01:01
3,KNQTWQIEKASLIEVKTCL,206,224,1,0.08,HLA-DRB1*01:01
4,KNQTWQIEKASLIEVKTCLW,206,225,1,0.11,HLA-DRB1*01:01
...,...,...,...,...,...,...
75,YRQGYATQTVGPWHLGKLEID,256,276,1,0.86,HLA-DQA1*05:01/DQB1*02:01
76,HRLMSAAIKDQKAVHADMG,181,199,1,0.87,HLA-DRB4*01:01
77,GSGIFVVDNVHTWTEQYKF,16,34,1,0.90,HLA-DRB3*01:01
78,QKAVHADMGYWIES,191,204,1,0.95,HLA-DRB3*01:01


## Separando epítopos e criando arquivo FASTA para IEDB analysis resource

In [ ]:
pepitides = epitopos_repetidos.peptide

with open("./peptideos_tcell_2.fasta", "w") as f:
    for i, peptide in enumerate(pepitides, start=1):
        f.write(f">NP {i}\n")
        f.write(f"{peptide}\n")
        
pepitides

0       YRQGYATQTVGPWH
1        QYKFQPESPARLA
2        YRQGYATQTVGPW
3       KNQTWQIEKASLIE
4    KNQTWQIEKASLIEVKT
5      QYKFQPESPARLASA
6       QYKFQPESPARLAS
7     KNQTWQIEKASLIEVK
8        AAIKDQKAVHADM
9        HRLMSAAIKDQKA
Name: peptide, dtype: str

### Seqkit remove sequências proteicas contendo gaps e *.

In [10]:
!seqkit grep -s -v -r -p '[-*X?]' './Fastas/denv4_NS1_final.fasta' > DENV4_seq_filter_all.fasta

### Resultado IEDB analysis resource

In [11]:
conservacy_result = pd.read_csv('./ConservancyResult_tcell_2.csv')
conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,View details
0,1,NP 1,YRQGYATQTVGPW,13,62.42% (93/149),84.62%,100.00%,NaN
1,2,NP 2,HRLMSAAIKDQKA,13,95.30% (142/149),76.92%,100.00%,NaN
2,3,NP 3,AKIFTPEARNSTF,13,71.81% (107/149),76.92%,100.00%,NaN
3,4,NP 4,YRQGYATQTVGPWHL,15,62.42% (93/149),86.67%,100.00%,NaN
4,5,NP 5,HRLMSAAIKDQKAVHADMGY,20,95.30% (142/149),85.00%,100.00%,NaN
...,...,...,...,...,...,...,...,...
114,115,NP 115,SECPNERRAWNSLEVEDY,18,29.53% (44/149),83.33%,100.00%,NaN
115,116,NP 116,QYKFQPESPARLASAILNAH,20,88.59% (132/149),80.00%,100.00%,NaN
116,117,NP 117,GIRSTTRLENVMWKQITNEL,20,83.89% (125/149),85.00%,100.00%,NaN
117,118,NP 118,GDVKGVLTKGKRALTPPVSDL,21,24.16% (36/149),71.43%,100.00%,NaN


### Merge da colunas qte_de_alelos e alelos ao dataframe conservacy_result

In [12]:
# qte_de_alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "qte_de_alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("View details"))

# alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("Epitope #"))

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos
0,NP 1,YRQGYATQTVGPW,13,62.42% (93/149),84.62%,100.00%,2.0,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1..."
1,NP 2,HRLMSAAIKDQKA,13,95.30% (142/149),76.92%,100.00%,2.0,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*01:02/DQB1..."
2,NP 3,AKIFTPEARNSTF,13,71.81% (107/149),76.92%,100.00%,NaN,NaN
3,NP 4,YRQGYATQTVGPWHL,15,62.42% (93/149),86.67%,100.00%,NaN,NaN
4,NP 5,HRLMSAAIKDQKAVHADMGY,20,95.30% (142/149),85.00%,100.00%,NaN,NaN
...,...,...,...,...,...,...,...,...
114,NP 115,SECPNERRAWNSLEVEDY,18,29.53% (44/149),83.33%,100.00%,NaN,NaN
115,NP 116,QYKFQPESPARLASAILNAH,20,88.59% (132/149),80.00%,100.00%,NaN,NaN
116,NP 117,GIRSTTRLENVMWKQITNEL,20,83.89% (125/149),85.00%,100.00%,NaN,NaN
117,NP 118,GDVKGVLTKGKRALTPPVSDL,21,24.16% (36/149),71.43%,100.00%,NaN,NaN


### Gerando a coluna percent_match para filtrar os epitopos com percentagem de match maior que 50%

In [13]:
col = "Percent of protein sequence matches at identity <= 100%"

conservacy_result["percent_match"] = (
    conservacy_result[col]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos,percent_match
0,NP 1,YRQGYATQTVGPW,13,62.42% (93/149),84.62%,100.00%,2.0,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1...",62.42
1,NP 2,HRLMSAAIKDQKA,13,95.30% (142/149),76.92%,100.00%,2.0,"HLA-DPA1*02:01/DPB1*14:01, HLA-DQA1*01:02/DQB1...",95.30
2,NP 3,AKIFTPEARNSTF,13,71.81% (107/149),76.92%,100.00%,NaN,NaN,71.81
3,NP 4,YRQGYATQTVGPWHL,15,62.42% (93/149),86.67%,100.00%,NaN,NaN,62.42
4,NP 5,HRLMSAAIKDQKAVHADMGY,20,95.30% (142/149),85.00%,100.00%,NaN,NaN,95.30
...,...,...,...,...,...,...,...,...,...
114,NP 115,SECPNERRAWNSLEVEDY,18,29.53% (44/149),83.33%,100.00%,NaN,NaN,29.53
115,NP 116,QYKFQPESPARLASAILNAH,20,88.59% (132/149),80.00%,100.00%,NaN,NaN,88.59
116,NP 117,GIRSTTRLENVMWKQITNEL,20,83.89% (125/149),85.00%,100.00%,NaN,NaN,83.89
117,NP 118,GDVKGVLTKGKRALTPPVSDL,21,24.16% (36/149),71.43%,100.00%,NaN,NaN,24.16


### Sort e filtragem por percent_match

In [ ]:
# Filtro do Percent match
conservacy_result_filtered = (
    conservacy_result[conservacy_result["percent_match"] >= parametros.percent_match_minimo_tcc2]
    .sort_values(
            by="percent_match", 
            ascending=False
        )
    ).reset_index(drop=True)

conservacy_result_filtered

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos,percent_match
0,NP 64,QYKFQPESPSKLAS,14,99.19% (859/866),92.86%,100.00%,7,"HLA-DPA1*02:01/DPB1*01:01, HLA-DPA1*03:01/DPB1...",99.19
1,NP 97,QYKFQPESPSKLA,13,99.19% (859/866),92.31%,100.00%,9,"HLA-DPA1*02:01/DPB1*01:01, HLA-DPA1*03:01/DPB1...",99.19
2,NP 47,QYKFQPESPSKLASA,15,99.19% (859/866),93.33%,100.00%,7,"HLA-DPA1*02:01/DPB1*01:01, HLA-DPA1*03:01/DPB1...",99.19
3,NP 107,HTWTEQYKFQPESPSKLASA,20,99.19% (859/866),95.00%,100.00%,6,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01, HLA...",99.19
4,NP 137,HTWTEQYKFQPESPSKLAS,19,99.19% (859/866),94.74%,100.00%,6,"HLA-DRB1*01:01, HLA-DRB1*04:01, HLA-DRB1*04:05...",99.19
5,NP 85,QYKFQPESPSKLASAI,16,98.96% (857/866),93.75%,100.00%,5,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01, HLA...",98.96
6,NP 125,HTWTEQYKFQPESPSKLASAI,21,98.96% (857/866),95.24%,100.00%,6,"HLA-DPA1*03:01/DPB1*04:02, HLA-DRB1*01:01, HLA...",98.96
7,NP 119,ADMGYWIESALNDTWK,16,97.23% (842/866),68.75%,100.00%,7,"HLA-DPA1*01:03/DPB1*04:01, HLA-DPA1*02:01/DPB1...",97.23
8,NP 126,ADMGYWIESALNDT,14,97.23% (842/866),71.43%,100.00%,5,"HLA-DPA1*01:03/DPB1*04:01, HLA-DPA1*02:01/DPB1...",97.23
9,NP 98,GSGIFITDNVHTWTE,15,97.00% (840/866),93.33%,100.00%,7,"HLA-DQA1*03:01/DQB1*03:02, HLA-DQA1*05:01/DQB1...",97.00
